# Secure Data Sharing

 ## Sharing Options

 - Listing, in which you offer a share and additional metadata as a data product to one or more accounts,
 - Direct Share, in which you directly share specific database objects (a share) to another account in your region,
 - Data Exchange, in which you set up and manage a group of accounts and offer a share to that group,
 - clean room, in which you can share data and control which queries can be run against our data.

You can share the following Snowflake objects:

- Databases
- Tables
- Dynamic tables
- External tables
- Externally managed and managed Apache Iceberg™ tables
- Externally managed Delta Lake tables (with Delta Direct and catalog-linked databases)
- Views:
    - Regular views
    - Secure views
    - Secure materialized views
    - Semantic views

- Cortex Search services
- User-defined functions (UDFs) (secure and non-secure)
- Models of type USER_MODEL, CORTEX_FINETUNED, or DOC_AI


 - Virtual warehouse is not part of a share. 
 - If a Snowflake customer consumes a share, they will use their own virtual warehouse.
 - If a non-Snowflake customer is consuming the Share, they will use the data provider compute through a data provider-created virtual warehouse (which would have been separately configured)
 - Consumer cannot use Time Travel on shared data

## Costs
 - no actual data is copied or transferred between accounts. 
 - Shared data does not take up any storage in a consumer account and therefore does not contribute to the consumer’s monthly data storage charges. 
 - The only charges to consumers are for the compute resources (i.e. virtual warehouses) used to query the imported data.


 ## How works

 ![alt](https://docs.snowflake.com/en/_images/data-sharing-overview.png)



 

In [ ]:
Use role sysadmin;
CREATE OR REPLACE DATABASE DATA_S;
use DATA_S.public;
CREATE OR REPLACE STAGE aws_stage
    url='s3://bucketsnowflakes3';

// List files in stage
LIST @aws_stage;


In [ ]:
// Create table
CREATE OR REPLACE TABLE DATA_S.public.ORDERS (
ORDER_ID	VARCHAR(30)
,AMOUNT	NUMBER(38,0)
,PROFIT	NUMBER(38,0)
,QUANTITY	NUMBER(38,0)
,CATEGORY	VARCHAR(30)
,SUBCATEGORY	VARCHAR(30));

In [ ]:
// Load data using copy command
COPY INTO DATA_S.public.ORDERS
    FROM @DATA_S.public.aws_stage
    file_format= (type = csv field_delimiter=',' skip_header=1)
    pattern='.*OrderDetails.*';


In [ ]:
SELECT * FROM DATA_S.public.ORDERS;

In [ ]:
CREATE OR REPLACE SECURE VIEW ORDERS_VIEW_SECURE AS
SELECT 
ORDER_ID,
AMOUNT,
QUANTITY
FROM ORDERS
WHERE CATEGORY != 'Furniture'; 

In [ ]:
-- Create a share object

-- You need the ACCOUNTADMIN role or Create Share 
USE ROLE ACCOUNTADMIN;

-- Create Share
CREATE OR REPLACE SHARE ORDERS_SHARE;


In [ ]:
--Setup Grants 

// Grant usage on database
GRANT USAGE ON DATABASE DATA_S TO SHARE ORDERS_SHARE; 
// Grant usage on schema
GRANT USAGE ON SCHEMA DATA_S.PUBLIC TO SHARE ORDERS_SHARE; 
// Grant SELECT on table
GRANT SELECT ON TABLE DATA_S.PUBLIC.ORDERS TO SHARE ORDERS_SHARE; 
// Grant select on view
GRANT SELECT ON VIEW  DATA_S.PUBLIC.ORDERS_VIEW_SECURE TO SHARE ORDERS_SHARE;


-- "When sharing data in Snowflake, the Provider (the account sharing the data) must grant the following privileges:
-- SELECT on the specific tables in the database
-- USAGE on the database and schema"


// Validate Grants
SHOW GRANTS TO SHARE ORDERS_SHARE;

## Reader accounts

- Belongs to the provider account that created it. 
- Provider share databases with reader accounts;
- reader account can ***only*** consume data from the provider account that created it.

Refer to the following diagram:

![alt](https://docs.snowflake.com/en/_images/data-sharing-reader.png)

 - Users in a reader account can query data that has been imported with the reader account.
 - but cannot perform any of the DML tasks that are allowed in a full account, such as:
    - data loading
    - insert
    - update

In [ ]:
-- Create Reader Account --

CREATE MANAGED ACCOUNT reader_account
ADMIN_NAME = read_acc_admin,
ADMIN_PASSWORD = 'Password-123456',
TYPE = READER;


In [ ]:
--- To drop the account again: DROP MANAGED ACCOUNT reader_account;

// Show accounts
SHOW MANAGED ACCOUNTS;
-- allows providers to manage and track the reader accounts they have created by using the SHOW MANAGED ACCOUNTS command, ensuring visibility into all reader accounts associated with a provider. Reader account has no data of its own.


In [ ]:

-- Share the data -- 

ALTER SHARE ORDERS_SHARE 
ADD ACCOUNT = VFB86606;

-- -- Sharing to a lower edition
-- ALTER SHARE ORDERS_SHARE 
-- ADD ACCOUNT =  VFB86606
-- SHARE_RESTRICTIONS=false;

In [ ]:
//// STEP 4:Create database from share ////
--- By using reader account ---
-- IMPORT SHARE. This privilege allows the user to import shared data from another account, which is necessary when accessing data shared through the Snowflake Marketplace.

// Show all shares (consumer & producers)
SHOW SHARES;

// See details on share
DESC SHARE <consumer_account>.ORDERS_SHARE;

// Create a database in consumer account using the share
CREATE DATABASE DATA_SHARE_DB FROM SHARE <account_producer>.ORDERS_SHARE;

// Validate table access
SELECT * FROM  DATA_SHARE_DB.PUBLIC.ORDERS_VIEW_SECURE;


// Setup virtual warehouse
CREATE WAREHOUSE READ_WH WITH
WAREHOUSE_SIZE='X-SMALL'
AUTO_SUSPEND = 180
AUTO_RESUME = TRUE
INITIALLY_SUSPENDED = TRUE;


## Data Exchange

Provides a data hub for securely collaborating around data with a selected group of members that you invite
![alt](https://docs.snowflake.com/en/_images/private-data-exchange-govern.png)

you can easily provide data to a specific group of consistent business partners taking part in the Data Exchange, such as internal departments in your company or vendors, suppliers, and partners external to your company. If you want to share data with a variety of consumers inside and outside your organization, you can also use listings offered to specific consumers or publicly on the Snowflake Marketplace.

"The necessary privileges for a consumer in the Data Exchange to make a request and receive data are CREATE DATABASE and IMPORT SHARE. These privileges allow the consumer to create a database from the shared data and import the data share into their environment for use.



## Data Sharing Usage
display information about listings published in the Snowflake Marketplace or a data exchange. This includes telemetry data (number of clicks), as well as consumption data (queries run by consumers)

In [ ]:
select * from snowflake.data_sharing_usage.monetized_usage_daily;

In [ ]:
select * from snowflake.data_sharing_usage.listing_telemetry_daily;

In [ ]:
select * from snowflake.data_sharing_usage.APPLICATION_STATE;

## Data Clean Rooms

- Not available in government and VPS deployments.
Data clean rooms are configurable, isolated Snowflake environments where collaborators can import data, specify what queries can be run against that data, and configure data protection settings such as differential privacy and specifying joinable and projectable columns. Access to a clean room is by invitation only.

- Clean rooms don’t support monetization features

https://www.youtube.com/watch?v=FC4Ug95vepM


## listings

"For all listings shared with specific consumer accounts, Snowsight automatically detects if the target account is in a different region and enables auto-fulfillment. It is not possible to manually replicate private listings to other regions.

For more detailed information, refer to the official Snowflake documentation."

"The ORGADMIN role is responsible for accepting the Snowflake Consumer Terms of Service, as this role manages organization-wide settings and agreements, including access to the Snowflake Marketplace.

For more detailed information, refer to the official Snowflake documentation."
